# 03 — Evolved (neural) genomes

MechaTree's `safety` and `allocation` decisions can come from one of two
places:

- A **constant genome** — a fixed scalar `safety = 3.0` and fixed
  allocation fractions. This is the default we used in notebook 01.
- A **neural genome** — the three-layer tanh networks that Eloy et al.
  2017 evolved using a population-level tournament. The state of each
  branch (number of leaves, peak stress) is fed into the network at
  every generation, and the output sets that branch's growth target.

Two distinct clusters survived the tournament — see fig. 7 of the
paper. We call them *species 0* ("interior") and *species 1*
("periphery") after their position in the trait scatter. The champion
weights live in [`data/S3_champions.json`](../data/S3_champions.json),
extracted from the Fortran tournament dump.

This notebook grows three trees from the same seed and compares them
side by side.

In [ ]:
from pathlib import Path

import numpy as np

from mechatree.config import load_config
from mechatree.genome import ConstantAllocation, ConstantSafety, load_champion
from mechatree.plotting import figstyle, plot_tree_3d
from mechatree.simulate import grow_tree

figstyle.apply()

cfg = load_config(Path("../examples/forest.yaml").resolve())
champions_path = Path("../data/S3_champions.json").resolve()

## Load the champions

`load_champion(path, species_id)` returns `(NeuralSafety,
NeuralAllocation, metadata)`. The metadata dict carries bookkeeping
fields from the tournament — number of seeds dropped, moment on leaves,
cluster centroid — useful for sanity-checking which species you've
actually loaded.

In [ ]:
safety_0, alloc_0, angles_0, info_0 = load_champion(champions_path, species_id=0)
safety_1, alloc_1, angles_1, info_1 = load_champion(champions_path, species_id=1)

for sid, info, angles in [(0, info_0, angles_0), (1, info_1, angles_1)]:
    print(
        f"species {sid}: n_members={info['n_members']}  "
        f"centroid_tag={tuple(round(x, 3) for x in info['centroid_tag'])}  "
        f"champion_n_seeds={info['champion_n_seeds']}"
    )
    print(
        f"  angles: theta1={np.degrees(angles['theta1']):.1f}°  "
        f"theta2={np.degrees(angles['theta2']):.1f}°  "
        f"gamma={np.degrees(angles['gamma1']):.1f}°"
    )

## Grow three trees

Same RNG seed, same config — only the genome changes. The constant
genome reproduces the baseline behaviour from notebook 01.

In [ ]:
from dataclasses import replace

n_generations = 100
seed = 42

default_safety = ConstantSafety(cfg.genome.safety)
default_alloc = ConstantAllocation(
    p_seeds=cfg.genome.p_seeds,
    p_leaves=cfg.genome.p_leaves,
    phototropism=cfg.genome.phototropism,
)

# Each Neural champion carries its own (theta1, theta2, gamma1, gamma2)
# in genome[0..2] (Fortran mod_tree.f90:108-111). Apply them to cfg.tree
# before growing so the simulator uses the champion's branching geometry
# instead of the YAML defaults.
cfg_sp0 = replace(cfg, tree=replace(cfg.tree, **angles_0))
cfg_sp1 = replace(cfg, tree=replace(cfg.tree, **angles_1))

t_const = grow_tree(
    cfg, n_generations=n_generations, seed=seed, safety=default_safety, allocation=default_alloc
)
t_sp0 = grow_tree(
    cfg_sp0, n_generations=n_generations, seed=seed, safety=safety_0, allocation=alloc_0
)
t_sp1 = grow_tree(
    cfg_sp1, n_generations=n_generations, seed=seed, safety=safety_1, allocation=alloc_1
)

print(f"{'genome':>20}  {'branches':>10}  {'leaves':>8}  {'trunk d':>8}")
for label, tree in [
    ("constant (safety=3.0)", t_const),
    ("species 0 (interior)", t_sp0),
    ("species 1 (periphery)", t_sp1),
]:
    print(
        f"{label:>20}  "
        f"{tree.get_number_of_branches():>10}  "
        f"{tree.get_total_leaves():>8}  "
        f"{tree.get_diameter(0):>8.4f}"
    )

Even at this size the two species behave differently. The neural
networks adapt their safety target to the local branch state, so
trunks under heavy load thicken more aggressively than the constant
rule allows, while lightly loaded twigs spend less mass on wood.

## Render side by side

We re-use `plot_tree_3d` to draw each tree and copy the traces into a
1×3 grid of plotly 3D scenes.

In [ ]:
scenarios = [
    ("constant (safety=3)", t_const),
    ("species 0 (interior)", t_sp0),
    ("species 1 (periphery)", t_sp1),
]

fig = figstyle.subplots(
    size="full",
    aspect=3.0,
    rows=1,
    cols=3,
    specs=[[{"type": "scene"} for _ in scenarios]],
    subplot_titles=[label for label, _ in scenarios],
)
for col, (_, tree) in enumerate(scenarios, start=1):
    sub_fig = plot_tree_3d(tree)
    for trace in sub_fig.data:
        fig.add_trace(trace, row=1, col=col)
    scene_key = "scene" if col == 1 else f"scene{col}"
    fig.layout[scene_key].update(sub_fig.layout.scene)
fig.update_layout(showlegend=False)
fig.show()

## Loading the neural genome from YAML

Manual calls to `load_champion` are useful in notebooks, but for
end-to-end pipelines you can flip the genome directly in the YAML
config. Uncomment the `neural_from` block in `examples/forest.yaml`,
and `load_config` will hand `grow_tree` the matching `NeuralSafety` /
`NeuralAllocation` automatically.

```yaml
genome:
  neural_from:
    path: ../data/S3_champions.json
    species_id: 0
```

`path` resolves relative to the YAML file's location.

## Where to next

- [04_custom_growth_law.ipynb](04_custom_growth_law.ipynb) — sweep the
  constant genome parameters along with `wind_fn` and `Sun`.
- [05_strahler_diagnostics.ipynb](05_strahler_diagnostics.ipynb) — see
  whether the species' self-similar architecture differs as well as
  their overall shape.
- User guide: [Customizing the simulation](../docs/userguide.rst).